# Denoising — ANTsImage interface

Pass the real r16 tutorial image and its mask directly to
`antstorch.denoise_image`. The function returns an ANTsImage with the same
geometry. No artificial noise or pretrained weights are used.

Compare [the tensor interface](denoising_tensor.ipynb). This is one public
function dispatching on input type, rather than two different denoising methods.

The image is resized to 128 × 96, matching the tensor example and giving
unequal voxel spacings to make the array-axis convention explicit.

Install ANTsTorch and notebook tools from the repository root before opening this notebook:
```bash
python -m pip install -e '.[notebooks]'
python -m ipykernel install --user --name python3 --display-name "Python 3 (ANTsTorch)"
```
Select that kernel and run the cells in order. All examples default to CPU.

In [ ]:
from time import perf_counter
_example_started = perf_counter()

import torch  # Import before ANTs to avoid OpenMP conflicts on macOS.
import numpy as np
import ants
import antstorch

# Display figures inside Jupyter/VS Code even if MPLBACKEND was set to Agg.
from IPython import get_ipython
shell = get_ipython()
if shell is not None:
    shell.run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt

device = "cpu"
antstorch.set_default_device(device)

# Synchronize the selected accelerator so timings include completed GPU work.
from contextlib import contextmanager
_example_operation_times = []

def _synchronize_example_device():
    selected = torch.device(device)
    if selected.type == "cuda":
        torch.cuda.synchronize(selected)
    elif selected.type == "mps":
        torch.mps.synchronize()

@contextmanager
def time_step(label):
    _synchronize_example_device()
    started = perf_counter()
    try:
        yield
    finally:
        _synchronize_example_device()
        seconds = perf_counter() - started
        _example_operation_times.append((label, seconds))
        print(f"{label}: {seconds:.4f} s")

In [ ]:
image = ants.image_read(ants.get_ants_data("r16")).clone("float")
image = ants.resample_image(image, (128, 96), use_voxels=True)
mask = ants.get_mask(image)
with time_step('Denoising (denoised)'):
    denoised = antstorch.denoise_image(
        image, mask=mask, noise_model="Rician", p=1, r=2, shrink_factor=1, device=device,
    )
original = image.numpy()
output = denoised.numpy()
assert output.shape == original.shape and np.isfinite(output).all()
for attribute in ("spacing", "origin", "direction"):
    np.testing.assert_allclose(getattr(denoised, attribute), getattr(image, attribute))
residual = original - output
inside = mask.numpy() > 0
print("Mean absolute intensity change inside mask:", float(np.abs(residual[inside]).mean()))

## Inspect filtered anatomy and the residual
A Rician noise model is selected for this magnitude-MRI example; assess whether
it fits your acquisition. Patch radius `p` and search radius `r` affect smoothing,
detail preservation and run time. The residual can contain anatomical detail as
well as noise and must not be treated as a ground-truth noise map.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3), constrained_layout=True)
for ax, array, title in zip(axes, [original, output, residual],
                            ["Tutorial r16", "Denoised", "Removed residual"]):
    view = ax.imshow(array.T, origin="lower", cmap="gray",
                     vmin=None if title == "Removed residual" else original.min(),
                     vmax=None if title == "Removed residual" else original.max())
    ax.set_title(title); ax.axis("off")
    fig.colorbar(view, ax=ax, shrink=0.7)
plt.show()

Inspect edges and small structures after filtering. Reduced residual variance alone does not establish preserved anatomy.

## Execution time

Run **Restart Kernel and Run All Cells** for a comparable end-to-end duration.
The total starts at the first setup cell and includes imports, downloads, data
preparation, computation, plotting and file writes. Kernel startup is excluded;
manual pauses between cells are included. Operation timings include any
preprocessing or asset downloads performed inside the called function. First
runs can be slower because caches are empty. Timings describe this run on the
selected device, not a fixed performance estimate for other machines.

In [ ]:
_synchronize_example_device()
total_seconds = perf_counter() - _example_started
operation_seconds = sum(seconds for _, seconds in _example_operation_times)
print(f"Total elapsed time: {total_seconds:.2f} s ({total_seconds / 60:.2f} min)")
print(f"Timed operations: {operation_seconds:.4f} s on {device}")